In [ ]:
import sys
from pathlib import Path
import json

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from sklearn.linear_model import RidgeCV

from src.modeling import *
from src.data_split import *
from src.preprocessing import *

In [ ]:
train_df = pd.read_csv('../data/Raw_data/train.csv', index_col='id', parse_dates=['timestamp'])
test_df = pd.read_csv('../data/Raw_data/test.csv', index_col='id', parse_dates=['timestamp'])
test_ids = pd.read_csv('../data/Raw_data/test.csv', parse_dates=['timestamp'])['id']
marco_df = pd.read_csv('../data/Raw_data/macro.csv', parse_dates=['timestamp'])
train_df = train_df.merge(marco_df, on="timestamp", how="left")
test_df = test_df.merge(marco_df, on="timestamp", how="left")

In [ ]:
with open("../data/features/features_100.json", "r", encoding="utf-8") as f:
    selected_features = json.load(f)

with open("../data/best_params/alpha_staking.json", "r", encoding="utf-8") as f:
    alpha_staking = json.load(f)

with open("../data/best_params/cat_params.json", "r", encoding="utf-8") as f:
    cat_params = json.load(f)

with open("../data/best_params/lgb_params.json", "r", encoding="utf-8") as f:
    lgb_params = json.load(f)

with open("../data/best_params/xgb_params.json", "r", encoding="utf-8") as f:
    xgb_params = json.load(f)


In [ ]:
X_train, X_test = prep_with_features_eng(train_df), prep_with_features_eng(test_df)
y_train = X_train['price_doc']
X_train, X_test = X_train[selected_features], X_test[selected_features]

In [ ]:
model_xgb = XGBRegressor(**xgb_params,
                    eval_metric =  "rmse",
                    tree_method = "hist",
                    objective = 'reg:squarederror',
                    enable_categorical = True,
                    random_state = 42,
                    n_jobs = -1,)
model_xgb.fit(X_train,
              y_train,
              verbose=200)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.42742090916152475
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import

In [ ]:
xgb_pred_log = model_xgb.predict(X_test)
xgb_pred = np.expm1(xgb_pred_log)
xgb_submit = pd.DataFrame({'id': test_ids, 'price_doc': xgb_pred})
xgb_submit.to_csv('../data/submissions/xgb_100_tuned.csv', index=False)

In [ ]:
X_train_cat = X_train.copy()
X_test_cat = X_test.copy()

cat_features = X_train.select_dtypes(
            include="category"
            ).columns.to_list()

for col in cat_features:
    X_train_cat[col] = X_train_cat[col].astype(str).fillna('__MISSING__')
    X_test_cat[col] = X_test_cat[col].astype(str).fillna('__MISSING__')


model_cat = CatBoostRegressor(**cat_params,
                        eval_metric = "RMSE",
                        loss_function = 'RMSE',
                        random_seed = 42,
                        verbose = 200)

model_cat.fit(X_train_cat,
            y_train,
            cat_features=cat_features)


You should provide test set for use best model. use_best_model parameter has been switched to false value.


0:	learn: 0.6018670	total: 251ms	remaining: 5m 28s
200:	learn: 0.4660446	total: 19.5s	remaining: 1m 48s
400:	learn: 0.4488494	total: 38.6s	remaining: 1m 27s
600:	learn: 0.4372418	total: 57.8s	remaining: 1m 8s
800:	learn: 0.4247891	total: 1m 17s	remaining: 49.7s
1000:	learn: 0.4114018	total: 1m 38s	remaining: 30.6s
1200:	learn: 0.3981387	total: 1m 58s	remaining: 11s
1311:	learn: 0.3910348	total: 2m 9s	remaining: 0us


CatBoostRegressor(bagging_temperature=0.5708563287223052, border_count=233, depth=8, eval_metric='RMSE', iterations=1312, l2_leaf_reg=3.455501005045953e-06, learning_rate=0.015017092532527218, loss_function='RMSE', random_seed=42, random_strength=3.4893013333133185, verbose=200)

In [ ]:
cat_pred_log = model_cat.predict(X_test_cat)
cat_pred = np.expm1(cat_pred_log)
cat_submit = pd.DataFrame({'id': test_ids, 'price_doc': cat_pred})
cat_submit.to_csv('../data/submissions/cat_100_tuned.csv', index=False)

In [ ]:
model_lgb = LGBMRegressor(**lgb_params,
                    objective = 'regression',
                    min_child_samples = 20,
                    random_state = 42,
                    n_jobs = -1,
                    verbose = 200)

model_lgb.fit(X_train,
              y_train,
              eval_metric="rmse",
              categorical_feature=cat_features)

[LightGBM] [Debug] Dataset::GetMultiBinFromSparseFeatures: sparse rate 0.819359
[LightGBM] [Debug] Dataset::GetMultiBinFromAllFeatures: sparse rate 0.075118
[LightGBM] [Debug] init for col-wise cost 0.002047 seconds, init for row-wise cost 0.014213 seconds
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.016267 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 20178
[LightGBM] [Info] Number of data points in the train set: 30461, number of used features: 100
[LightGBM] [Info] Start training from score 15.609548
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 8 and depth = 3
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves = 8 and depth = 3
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Debug] Trained a tree with leaves

,boosting_type,'gbdt'
,num_leaves,19
,max_depth,3
,learning_rate,0.014624076339413969
,n_estimators,1580
,subsample_for_bin,200000
,objective,'regression'
,class_weight,None
,min_split_gain,0.364877203684999
,min_child_weight,1
,min_child_samples,20


In [ ]:
lgb_pred_log = model_lgb.predict(X_test)
lgb_pred = np.expm1(lgb_pred_log)
lgb_submit = pd.DataFrame({'id': test_ids, 'price_doc': lgb_pred})
lgb_submit.to_csv('../data/submissions/lgb_100_tuned.csv', index=False)

In [ ]:
test_stacking = pd.DataFrame({'catboost_100_tuned_pred': cat_pred_log, 'lightgbm_100_tuned_pred': lgb_pred_log, 'xgboost_100_tuned_pred': xgb_pred_log})

In [ ]:
oof_train_stacking = pd.read_csv('../data/journals/oof_stacking.csv', index_col=0)
oof_train_stacking

,catboost_100_tuned_pred,xgboost_100_tuned_pred,lightgbm_100_tuned_pred,y_true
0,15.923688,15.910771,15.903809,15.961740
1,16.089383,16.023457,16.002378,16.249124
2,16.730531,16.774868,16.605996,16.456067
3,15.487668,15.476921,15.462068,15.501061
4,15.248073,15.228682,15.227926,15.311108
...,...,...,...,...
30466,15.530609,15.550094,15.567313,15.725135
30467,15.742132,15.785863,15.728267,15.816991
30468,16.606112,16.697931,16.492907,17.034386
30469,15.504860,15.488005,15.478578,15.757264


In [ ]:
pred_cols = ['catboost_100_tuned_pred', 'xgboost_100_tuned_pred', 'lightgbm_100_tuned_pred']

X_meta = oof_train_stacking[pred_cols]
y_meta = oof_train_stacking["y_true"]

meta_model = RidgeCV(
        alpha=alpha_staking['alpha']
    )

meta_model.fit(X_meta, y_meta)
X_meta_test = test_stacking[pred_cols]

stack_pred_log = meta_model.predict(X_meta_test)
stack_pred = np.expm1(stack_pred_log)

In [ ]:
stack_submit = pd.DataFrame({'id': test_ids, 'price_doc': stack_pred})
stack_submit.to_csv('../data/submissions/staking_3_models.csv', index=False)

In [ ]:
model_cat.save_model("../data/models/catboost.cbm")
model_lgb.booster_.save_model("../data/models/lightgbm.txt")
model_xgb.save_model("../data/models/xgboost.json")

In [ ]:
import joblib
joblib.dump(meta_model, "../data/models/meta_model.pkl")

['../data/models/meta_model.pkl']